In [0]:
# Configuration
# Import the centralized config so we have all table names available
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"
exec(open(f"{UTILS_DIR}/config.py").read())

print("✅ Config loaded")

In [0]:
# Capture row counts BEFORE the second run
# We store the counts now so we can compare after running the pipeline again
# This is our "before" snapshot

# Dictionary mapping readable label to full table name
tables = {
    "bronze — raw_products"      : BRONZE_PRODUCTS,
    "bronze — raw_vendors"       : BRONZE_VENDORS,
    "silver — products"          : SILVER_PRODUCTS,
    "silver — vendors"           : SILVER_VENDORS,
    "silver — llm_extracted"     : LLM_EXTRACTED,
    "silver — taxonomy"          : SILVER_TAXONOMY,
    "silver — taxonomy_enriched" : TAXONOMY_ENRICHED,
    "gold   — product_summary"   : GOLD_SUMMARY,
}

# Capture counts before second run
counts_before = {}

print("=== ROW COUNTS BEFORE SECOND RUN ===")
for label, table in tables.items():
    # spark.table() reads the Delta table
    # .count() returns the number of rows
    count = spark.table(table).count()
    counts_before[label] = count
    print(f"  {label:<35} {count:>5} rows")

In [0]:
# Run the full pipeline a second time
# This is the core of the idempotency test
# We trigger the exact same pipeline that already ran once

BASE_PATH = "/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/pipeline"

print("Running pipeline for the second time...")
print("=" * 55)

# Each notebook is called in the same order as 00_run_pipeline
# timeout_seconds=600 gives each notebook up to 10 minutes to complete
dbutils.notebook.run(f"{BASE_PATH}/01_bronze_ingest",      timeout_seconds=300)
print("✅ Stage 1 — Bronze complete")

dbutils.notebook.run(f"{BASE_PATH}/02_silver_standardize", timeout_seconds=300)
print("✅ Stage 2 — Silver standardize complete")

dbutils.notebook.run(f"{BASE_PATH}/03_llm_extraction",     timeout_seconds=600)
print("✅ Stage 3 — LLM extraction complete")


# Wait for Groq rate limit quota to reset before running the Judge
# Both notebooks consume tokens — running them back to back hits the TPM limit
print("⏳ Waiting 200 seconds for API rate limit quota to reset...")
import time
time.sleep(200)


dbutils.notebook.run(f"{BASE_PATH}/04_llm_judge",          timeout_seconds=600)
print("✅ Stage 4 — LLM judge complete")

dbutils.notebook.run(f"{BASE_PATH}/05_silver_taxonomy",    timeout_seconds=300)
print("✅ Stage 5 — Silver taxonomy complete")

dbutils.notebook.run(f"{BASE_PATH}/06_gold_aggregation",   timeout_seconds=300)
print("✅ Stage 6 — Gold aggregation complete")

print("=" * 55)
print("Second run complete. Checking counts...")

In [0]:
# Capture row counts AFTER the second run
# This is our "after" snapshot
# We compare with counts_before to verify nothing was duplicated

counts_after = {}

print("=== ROW COUNTS AFTER SECOND RUN ===")
for label, table in tables.items():
    count = spark.table(table).count()
    counts_after[label] = count
    print(f"  {label:<35} {count:>5} rows")

In [0]:
# Compare before and after
# This is the actual idempotency assertion
# If counts match → pipeline is idempotent ✅
# If counts differ → pipeline has a duplication bug ❌

print("=== IDEMPOTENCY VALIDATION ===")
print(f"{'Table':<35} {'Before':>8} {'After':>8} {'Status':>10}")
print("-" * 65)

all_passed = True

for label in tables.keys():
    before = counts_before[label]
    after  = counts_after[label]

    # Compare before and after counts
    # If they match → idempotent for this table
    # If they differ → something was duplicated
    if before == after:
        status = "✅ OK"
    else:
        status = f"❌ DIFF (+{after - before})"
        all_passed = False  # mark overall test as failed

    print(f"  {label:<35} {before:>8} {after:>8} {status:>10}")

print("-" * 65)

# Final verdict
if all_passed:
    print("\n✅ IDEMPOTENCY CONFIRMED — Pipeline is safe to run multiple times")
    print("   All row counts are identical after second run.")
else:
    print("\n❌ IDEMPOTENCY FAILED — Some tables have different row counts")
    print("   Check tables marked with ❌ above.")

In [0]:

display(spark.table(LLM_EXTRACTED))